In [ ]:
#Imports:

import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from shapash import SmartExplainer

In [ ]:
#1. Carregamento de dados:

#Configuração automática do caminho
base_path = os.getcwd()
if os.path.basename(base_path) == 'notebooks':
    data_path = os.path.abspath(os.path.join(base_path, '..', 'data', 'processed', 'hydraulic_uci_processed.csv'))
else:
    data_path = os.path.abspath(os.path.join(base_path, 'data', 'processed', 'hydraulic_uci_processed.csv'))

print(f"Carregando dados para o Dashboard: {data_path}")

try:
    df = pd.read_csv(data_path)
except FileNotFoundError:
    print("Erro: Execute o notebook '05_etl_processamento_hidraulico.ipynb' primeiro")
    raise

#Preparação
y = df['Stable_Flag']
X = df.drop('Stable_Flag', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
#2. Treinamento do modelo:

print("Treinando modelo para o Dashboard...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
#3. Configuração do Dashboard

#Dicionário/Tradução:
features_dict = {
    'Pressure_1': 'Pressão Principal (bar)',
    'Pressure_2': 'Pressão Secundária (bar)',
    'Temperature_1': 'Temp. Tanque (°C)',
    'Vibration': 'Vibração (mm/s)',
    'Volume_Flow_1': 'Fluxo Principal (L/min)',
    'Cooling_Efficiency': 'Eficiência do Resfriador (%)',
    'Cooling_Power': 'Potência de Resfriamento (kW)',
    'Efficiency_Factor': 'Fator de Eficiência',
    'Stable_Flag': 'Status de Estabilidade'
}

xpl = SmartExplainer(
    model=model,
    features_dict=features_dict
)

xpl.compile(x=X_test, y_target=y_test)
print("Explicações compiladas com sucesso")

In [ ]:
#4. Execução:
print("Dashboard pronto")
print("Clique no link para abrir")

app = xpl.run_app(
    title_story="Monitoramento Hidráulico - Indústria 5.0",
    port=8050
)